# Medvlm-Image-Reliance on Colab

Runtime > Change runtime type > **GPU**. Add a Colab secret named `HF_TOKEN` (key icon, left bar) from an account that has accepted the MedGemma terms.
All state lives in git (manifests) and Drive (raw outputs), so a disconnect only costs the current chunk: just re-run the cells.

In [5]:
from google.colab import drive, userdata
import os
drive.mount('/content/drive')
os.environ['MEDVLM_OUTPUTS_DIR'] = '/content/drive/MyDrive/medvlm_outputs'   # survives disconnects
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.makedirs(os.environ['MEDVLM_OUTPUTS_DIR'], exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
REPO = 'https://github.com/thaitri2005/Medvlm-Image-Reliance.git'
if not os.path.exists('/content/Medvlm-Image-Reliance'):
    !git clone $REPO /content/Medvlm-Image-Reliance
%cd /content/Medvlm-Image-Reliance
!git pull --ff-only
!pip install -q -e ".[model,gpu,dev]"

/content/Medvlm-Image-Reliance
remote: Enumerating objects: 14, done.
remote: Counting objects: 100% (14/14), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 10 (delta 3), reused 10 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (10/10), 6.31 KiB | 3.15 MiB/s, done.
From https://github.com/thaitri2005/Medvlm-Image-Reliance
   fcd0845..24458b2  main       -> origin/main
Updating fcd0845..24458b2
Fast-forward
 .gitignore                  |   4 +-
 src/medvlm/data/__init__.py |   0
 src/medvlm/data/prepare.py  |  98 ++++++++++++++++++
 src/medvlm/data/sampling.py |  69 +++++++++++++
 src/medvlm/data/sources.py  | 236 ++++++++++++++++++++++++++++++++++++++++++++
 5 files changed, 405 insertions(+), 2 deletions(-)
 create mode 100644 src/medvlm/data/__init__.py
 create mode 100644 src/medvlm/data/prepare.py
 create mode 100644 src/medvlm/data/sampling.py
 create mode 100644 src/medvlm/data/sources.py
  Installing build dependencies ... done
  Checking if build 

In [7]:
PROFILE, MODEL = 'pilot', 'medgemma4b'
!medvlm doctor --model $MODEL

python      3.13.15 on Linux
torch       2.11.0+cu130  cuda=True
gpu         Tesla T4  14.6 GiB  bf16=True
transformers 5.18.0  (Gemma 3 / MedGemma needs >= 4.50)
hf access   OK for google/medgemma-4b-it
data dir    /content/Medvlm-Image-Reliance/data
outputs dir /content/drive/MyDrive/medvlm_outputs


## Gate: real-model integration tests (do this before any long run)
If these fail on MedGemma, fix the backend first (dtype, processor, padding). On a T4 try `--set model.dtype=float16` or `--set model.quantization=4bit`.

In [ ]:
!MEDVLM_TEST_MODEL=$MODEL pytest -m slow -x

In [ ]:
!medvlm prepare --profile $PROFILE --model $MODEL

## Paraphrases (first time only; the result is committed to git)

In [ ]:
!medvlm paraphrase --profile $PROFILE --model $MODEL
!medvlm audit-sheet --profile $PROFILE --model $MODEL -n 100

## Run (resumable; first measure throughput with `--limit 200`)

In [ ]:
!medvlm run --profile $PROFILE --model $MODEL --limit 200

In [ ]:
!medvlm run --profile $PROFILE --model $MODEL

In [ ]:
!medvlm analyze --profile $PROFILE --model $MODEL
import pandas as pd
pd.read_csv(f'results/{MODEL}/{PROFILE}/by_dataset.csv').round(3)

## Save what git should keep
Paraphrases and result tables are small. Download them (or push with a GitHub token stored as a Colab secret `GH_TOKEN`) and commit from your laptop.

In [ ]:
!zip -qr /content/drive/MyDrive/medvlm_keep.zip manifests results
print('saved to Drive: medvlm_keep.zip')